In [ ]:
import pandas as pd
import numpy as np
import pyterrier_deepct
import pyterrier as pt
if not pt.started():
  pt.init(boot_packages=["com.github.terrierteam:terrier-prf:-SNAPSHOT"])

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics2020.xml')
data = tree.getroot()
titles20 = []
titles21 = []
title_ids20 = []
title_ids21 = []
nums20 = []
nums21 = []
for topic in data:
    titles20.append(topic.find('title').text.lower()[:-1])
    num20 = topic.find('number').text
    nums20.append(num20)
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics.xml')
data = tree.getroot()    
for topic in data:
    titles21.append(topic.find('title').text.lower()[:-1])
    num21 = topic.find('number').text
    nums21.append(num21)
topics = pd.DataFrame()
topics['qid'] = nums20 + nums21
topics['query'] = titles20 + titles21

topics21 = pd.DataFrame()
topics21['qid'] = nums21
topics21['query'] = titles21

topics20 = pd.DataFrame()
topics20['qid'] = nums20
topics20['query'] = titles20

qrels_path=("/workspace/src/evaluate_targers/new_qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

qrels21_path=("/workspace/src/evaluate_targers/qrels.qrels")
qrels21 = pt.io.read_qrels(qrels21_path)

qrels20_path=("/workspace/src/evaluate_targers/qrels20.qrels")
qrels20 = pt.io.read_qrels(qrels20_path)

docnos = qrels.docno

In [ ]:
'''
Example of simple tuning bm25 and its not saving to the folder 
'''

import os
rootdir = "/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes"
# folders = ['conclusions','pool_topics','topics','topics_conclusion','topics_conclusion_leakage','topics_leakage','pool_topics_conclusions','conclusions_leakage','pools_conclusions']
folders = ['webd-dependency', 'combined-fasttext', 'essays-dependency',
          'original', 'essays-fasttext', 'ibm-fasttext', 'webd-fasttext']
objective="ndcg_cut_5"
i = 0
for subdir, dirs, files in os.walk(rootdir):
    for file in files:
        if(file == 'data.properties'):
            path = subdir + '/'+ file
            if subdir.endswith('100'):
                print(path)
                indexref = pt.IndexRef.of(path)
                index = pt.IndexFactory.of(indexref)
                dlm = pt.BatchRetrieve(index, wmodel="DirichletLM", controls={"dirichletlm.mu" : 2500})
                tuned_dlm, parameters = pt.KFoldGridSearch(
                    dlm,
                    {dlm: {"dirichletlm.mu" : list(range(0,10000,250))}},
                    [topics20, topics21],
                    [qrels20, qrels21],
                    objective
                )
                
                print(pt.Experiment([tuned_dlm], topics, qrels, names=[folders[i] + '_tuned_dlm_'], eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]))
                print(pt.Experiment([tuned_dlm], topics20, qrels, names=[folders[i] + '_tuned_dlm_20'], eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]))
                print(pt.Experiment([tuned_dlm], topics21, qrels, names=[folders[i] + '_tuned_dlm_21'], eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]))

                print(pt.Experiment([tuned_dlm[tuned_dlm['docno'].isin(docnos)]], topics, qrels, names=[folders[i] + '_tuned_dlm__unjudged'], eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]))
                print(pt.Experiment([tuned_dlm[tuned_dlm['docno'].isin(docnos)]], topics20, qrels, names=[folders[i] + '_tuned_dlm__unjudged20'], eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]))
                print(pt.Experiment([tuned_dlm[tuned_dlm['docno'].isin(docnos)]], topics21, qrels, names=[folders[i] + '_tuned_dlm__unjudged21'], eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]))

                i += 1
            